# 08 - SHAP Explainability Analysis

Using SHAP (SHapley Additive exPlanations) to interpret model predictions:

- Global feature importance via SHAP values
- Feature interaction and dependence analysis
- Individual prediction explanations (force plots)
- Multi-class classification SHAP analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from config import DATA_PROCESSED, FIGURES_DIR, MODELS_DIR, RANDOM_STATE, MODELS_TUNED

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
shap.initjs()

## 1. Load Test Data and Best Models

In [ ]:
df_test = pd.read_parquet(DATA_PROCESSED / 'test.parquet')
print(f"Test set: {df_test.shape}")

# Define feature columns (exclude identifiers, target, and derived columns)
exclude_cols = ['City', 'Date', 'AQI', 'AQI_Bucket', 'season',
                'AQI_Calculated', 'AQI_Bucket_Calculated', 'Dominant_Pollutant']
feature_cols = [c for c in df_test.columns if c not in exclude_cols]
target_col = 'AQI'

X_test = df_test[feature_cols].values
y_test = df_test[target_col].values
X_test = np.nan_to_num(X_test, nan=0.0)

print(f"Features: {len(feature_cols)}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
# Load best regression model (try tuned first, fallback to default)
try:
    reg_model = joblib.load(MODELS_DIR / 'tuned' / 'best_regression.joblib')
    reg_model_name = 'Best Tuned Regression'
    print(f"Loaded tuned regression model from {MODELS_DIR / 'tuned' / 'best_regression.joblib'}")
except FileNotFoundError:
    reg_model = joblib.load(MODELS_DIR / 'regression' / 'lightgbm.joblib')
    reg_model_name = 'LightGBM (default)'
    print(f"Loaded fallback regression model from {MODELS_DIR / 'regression' / 'lightgbm.joblib'}")

# Load best classification model
clf_model = joblib.load(MODELS_DIR / 'classification' / 'xgboost.joblib')
print(f"Loaded classification model from {MODELS_DIR / 'classification' / 'xgboost.joblib'}")

print(f"\nRegression model: {reg_model_name} ({type(reg_model).__name__})")
print(f"Classification model: XGBoost ({type(clf_model).__name__})")

## 2. SHAP TreeExplainer - Regression Model

In [ ]:
# Use a subsample for speed
shap_sample = X_test[:500]
print(f"SHAP sample size: {shap_sample.shape[0]}")

# Create TreeExplainer for regression model
explainer = shap.TreeExplainer(reg_model)
shap_values = explainer.shap_values(shap_sample)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base): {explainer.expected_value:.2f}")

## 3. Figure 26: SHAP Summary Plot (Beeswarm)

In [ ]:
fig = plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, shap_sample, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot (Beeswarm) - Regression Model', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '26_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / '26_shap_summary.png'}")

## 4. Figure 27: SHAP Bar Plot (Mean |SHAP|)

In [ ]:
fig = plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, shap_sample, feature_names=feature_cols,
                  plot_type="bar", show=False)
plt.title('Mean |SHAP Value| - Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '27_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / '27_shap_bar.png'}")

## 5. Figure 28: SHAP Dependence Plots (Top 5 Features)

In [ ]:
# Identify top 5 features by mean |SHAP value|
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top5_idx = np.argsort(mean_abs_shap)[::-1][:5]
top5_features = [feature_cols[i] for i in top5_idx]
print(f"Top 5 features by mean |SHAP|: {top5_features}")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, feat_idx in enumerate(top5_idx):
    plt.sca(axes[i])
    shap.dependence_plot(feat_idx, shap_values, shap_sample,
                         feature_names=feature_cols, ax=axes[i], show=False)
    axes[i].set_title(f'{feature_cols[feat_idx]}', fontsize=12)

# Hide the 6th subplot
axes[5].set_visible(False)

plt.suptitle('SHAP Dependence Plots - Top 5 Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '28_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / '28_shap_dependence.png'}")

## 6. Force Plots for Individual Predictions

In [ ]:
# Get predictions on the SHAP sample
preds_sample = reg_model.predict(shap_sample)

# Find indices for extreme predictions
good_idx = np.argmin(preds_sample)  # lowest predicted AQI ("Good")
severe_idx = np.argmax(preds_sample)  # highest predicted AQI ("Severe")

print(f"Good AQI prediction: index={good_idx}, predicted AQI={preds_sample[good_idx]:.1f}, actual AQI={y_test[good_idx]:.1f}")
print(f"Severe AQI prediction: index={severe_idx}, predicted AQI={preds_sample[severe_idx]:.1f}, actual AQI={y_test[severe_idx]:.1f}")

In [ ]:
# Force plot for "Good" AQI prediction
print("Force Plot: Good AQI Prediction (lowest predicted AQI)")
shap.force_plot(explainer.expected_value, shap_values[good_idx, :],
                shap_sample[good_idx, :], feature_names=feature_cols,
                matplotlib=True)

In [ ]:
# Force plot for "Severe" AQI prediction
print("Force Plot: Severe AQI Prediction (highest predicted AQI)")
shap.force_plot(explainer.expected_value, shap_values[severe_idx, :],
                shap_sample[severe_idx, :], feature_names=feature_cols,
                matplotlib=True)

## 7. SHAP for Classification Model

In [ ]:
# TreeExplainer on classification model (with fallbacks for XGBoost multiclass issues)
clf_model_used = 'XGBoost'
try:
    clf_explainer = shap.TreeExplainer(clf_model)
    clf_shap_values = clf_explainer.shap_values(shap_sample)
    print('TreeExplainer succeeded on XGBoost classifier.')
except Exception as e:
    print(f'TreeExplainer failed on XGBoost: {e}')
    print('Falling back to LightGBM classifier...')
    try:
        clf_model_fallback = joblib.load(MODELS_DIR / 'classification' / 'lightgbm.joblib')
        clf_explainer = shap.TreeExplainer(clf_model_fallback)
        clf_shap_values = clf_explainer.shap_values(shap_sample)
        clf_model_used = 'LightGBM'
        print('TreeExplainer succeeded on LightGBM classifier.')
    except Exception as e2:
        print(f'TreeExplainer failed on LightGBM: {e2}')
        print('Falling back to shap.Explainer with predict method...')
        background = shap.sample(shap_sample, min(100, len(shap_sample)))
        clf_explainer = shap.Explainer(clf_model.predict, background)
        clf_shap_values = clf_explainer(shap_sample).values
        clf_model_used = 'XGBoost (KernelExplainer)'
        print('shap.Explainer with predict method succeeded.')

if isinstance(clf_shap_values, list):
    print(f"Multi-class SHAP values ({clf_model_used}): {len(clf_shap_values)} classes")
    for i, sv in enumerate(clf_shap_values):
        print(f"  Class {i}: {sv.shape}")
else:
    print(f"SHAP values shape ({clf_model_used}): {clf_shap_values.shape}")

In [ ]:
# Figure 29: SHAP summary for multi-class classification
fig = plt.figure(figsize=(14, 10))

# Handle both list-of-arrays (multiclass) and single-array SHAP values
if isinstance(clf_shap_values, list):
    # For multiclass: summary_plot handles list of arrays natively
    shap.summary_plot(clf_shap_values, shap_sample, feature_names=feature_cols,
                      show=False)
elif clf_shap_values.ndim == 3:
    # 3D array (samples x features x classes) - convert to list of arrays
    shap_list = [clf_shap_values[:, :, i] for i in range(clf_shap_values.shape[2])]
    shap.summary_plot(shap_list, shap_sample, feature_names=feature_cols,
                      show=False)
else:
    # Single 2D array (e.g. from KernelExplainer on predict)
    shap.summary_plot(clf_shap_values, shap_sample, feature_names=feature_cols,
                      show=False)

plt.title(f'SHAP Summary - Classification Model ({clf_model_used})', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '29_shap_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / '29_shap_classification.png'}")

## Summary

SHAP analysis reveals:

1. **Global Feature Importance**: The beeswarm and bar plots show which features have the largest impact on AQI predictions across all test samples.
2. **Feature Effects**: Dependence plots illustrate how each top feature's value relates to its SHAP contribution, revealing non-linear relationships and interactions.
3. **Individual Explanations**: Force plots decompose individual predictions, showing exactly which features push the prediction higher or lower from the baseline.
4. **Classification Insights**: Multi-class SHAP analysis shows how features contribute differently to each AQI category prediction.

These interpretability results help validate that the model is learning meaningful environmental patterns rather than spurious correlations.